# 🚀 Automatisierter System- & Hardware-Check für NeMo / Automodel

Dieses Notebook prüft automatisch, ob dein Host-System alle Voraussetzungen (Festplattenspeicher, NVIDIA-Treiber, Docker & GPU-Passthrough) für den Container-Start erfüllt.

**Der Ablauf:**
Führe einfach die Zellen nacheinander aus. Wenn ein Check fehlschlägt, zeigt dir das Notebook gezielt an, was behoben werden muss.

In [ ]:
# Import benötigter Bibliotheken
import subprocess
import shutil
import sys
import os
import socket
from pathlib import Path

class HardwareCheckError(Exception):
    """Eigene Exception für Hardware-/Umgebungsfehler."""
    pass

print("✅ Python-Umgebung bereit.")

### Schritt 1: Festplattenspeicher prüfen
Überprüft, ob genügend Speicherplatz (z.B. für Container-Images und Modell-Caches) frei ist.

In [ ]:
def check_disk_space(required_gb=250):
    print("🔍 [1/4] Prüfe freien Festplattenspeicher...")
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024 ** 3)
    print(f"   --> Freier Speicherplatz: {free_gb:.2f} GB")
    
    if free_gb < required_gb:
        raise HardwareCheckError(
            f"Zu wenig Speicherplatz! Benötigt: mind. {required_gb} GB, Verfügbar: {free_gb:.2f} GB.\n"
            f"Tipp: Große Sprachmodelle und Container-Images benötigen viel Platz."
        )
    print("   ✅ Speicherkapazität ausreichend!\n")

try:
    check_disk_space(required_gb=250)
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}")

### Schritt 2: NVIDIA-Treiber & GPU-Verfügbarkeit prüfen
Überprüft, ob `nvidia-smi` auf dem Host reagiert und wie viel VRAM zur Verfügung steht.

In [ ]:
def check_nvidia_smi():
    print("🔍 [2/4] Prüfe Host-NVIDIA-Treiber & GPU-Verfügbarkeit...")
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True
        )
        gpus = result.stdout.strip().split("\n")
        print(f"   --> Gefundene GPUs ({len(gpus)}):")
        
        total_free_vram = 0
        for gpu in gpus:
            idx, name, mem_total, mem_free = [x.strip() for x in gpu.split(",")]
            free_vram = int(mem_free) / 1024
            total_free_vram += free_vram
            print(f"      • GPU {idx}: {name} | Freier VRAM: {free_vram:.2f} GB / {int(mem_total)/1024:.2f} GB")
            
        print(f"   --> Gesamter freier VRAM über alle GPUs: {total_free_vram:.2f} GB")
        if total_free_vram < 40:
            print("   ⚠️ WARNUNG: Der freie VRAM könnte für große LLMs knapp werden!")
        print("   ✅ Host GPU-Treiber erreichbar!\n")
        
    except FileNotFoundError:
        raise HardwareCheckError("'nvidia-smi' wurde nicht gefunden. Bitte installiere die NVIDIA-Treiber auf dem Host.")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(f"Fehler bei der Ausführung von nvidia-smi: {e.stderr}")

try:
    check_nvidia_smi()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Bitte behebe das Treiber-Problem, bevor du fortfährst.")

### Schritt 3: Docker-Installation & Daemon prüfen
Prüft, ob der Docker-Dienst läuft und der User die nötigen Berechtigungen hat.

In [ ]:
def check_docker_installation():
    print("🔍 [3/4] Prüfe Docker-Installation & Daemon-Status...")
    try:
        subprocess.run(["docker", "info"], capture_output=True, text=True, check=True)
        print("   ✅ Docker ist installiert und der Daemon läuft!\n")
    except FileNotFoundError:
        raise HardwareCheckError("Docker CLI nicht gefunden. Ist Docker auf dem System installiert?")
    except subprocess.CalledProcessError:
        raise HardwareCheckError("Docker-Daemon läuft nicht oder der aktuelle Benutzer hat keine Rechte (sudo / docker group).")

try:
    check_docker_installation()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Starte den Docker-Service oder prüfe deine Benutzerrechte.")

### Schritt 4: NVIDIA Container Toolkit prüfen (GPU-Passthrough)
Testet, ob ein Docker-Container erfolgreich auf die GPU zugreifen kann.

In [ ]:
def check_nvidia_container_toolkit():
    print("🔍 [4/4] Prüfe NVIDIA Container Toolkit (GPU-Passthrough in Docker)...")
    cmd = [
        "docker", "run", "--rm", "--gpus", "all",
        "nvidia/cuda:12.0.0-base-ubuntu22.04",
        "nvidia-smi"
    ]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("   ✅ GPU-Durchreichung an Docker-Container funktioniert einwandfrei!\n")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(
            f"Docker kann nicht auf die GPUs zugreifen! Fehler:\n{e.stderr}\n"
            "Tipp: Stelle sicher, dass das 'nvidia-container-toolkit' installiert ist und Docker neu gestartet wurde."
        )

try:
    check_nvidia_container_toolkit()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Installiere das NVIDIA Container Toolkit (siehe README).")

### Schritt 5: Projekt-Verzeichnisse & Umgebung vorbereiten
Wenn alle oberen Checks erfolgreich waren, richten wir die Ordner und optional das `.env`-Token ein.

In [ ]:
def setup_project_environment():
    print("=== Bereite Projekt-Umgebung vor ===")
    dirs = ["hf-cache", "data/raw", "data/sft", "results", "configs"]
    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)
        print(f"📁 Ordner bereit: {d}/")
        
    token = input("Gib deinen Hugging Face Token ein (optional, Enter zum Überspringen): ").strip()
    if token:
        Path(".env").write_text(f"HF_TOKEN={token}\n")
        print("✅ .env Datei erstellt.")
    else:
        print("ℹ️ Kein Token angegeben (wird übersprungen).")

setup_project_environment()

### Schritt 6: Container starten & Zugriffs-Anleitung anzeigen
Startet den gesamten Stack im Hintergrund und ermittelt automatisch die Server-IP sowie die Befehle für den Zugriff.

In [ ]:
def get_host_ip():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except Exception:
        return "localhost"

def start_stack_and_show_guide():
    print("=== Starte Docker Compose Stack ===")
    try:
        res = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True, check=True)
        print("🎉 Alle Container erfolgreich gestartet!\n")
        
        server_ip = get_host_ip()
        
        print("=" * 65)
        print(" 🌐 ZUGRIFFS- UND LÄUFIGKEITS-GUIDE")
        print("=" * 65)
        
        print("\n1. 🤖 NeMo-Container:
  - Zugriff per Terminal (SSH/Host):
    👉 docker exec -it nemo-24.09 bash
  - Jupyter Notebook darin starten (falls benötigt):
    👉 jupyter notebook --ip=0.0.0.0 --port=8889 --no-browser --allow-root
  - Funktionsprüfung: Öffne die URL im Browser. Das Jupyter-Interface sollte laden.")")

        print(f"\n2. 🧠 NeMo Automodel Container:
  - Zugriff per Terminal (SSH/Host):
    👉 docker exec -it nemo-automodel:26.06.00 bash
  - Jupyter Notebook darin starten (falls benötigt):
    👉 jupyter notebook --ip=0.0.0.0 --port=8887 --no-browser --allow-root
  - Web-Adresse: http://{server_ip}:8887 (oder http://localhost:8887)
  - Funktionsprüfung: Öffne die URL im Browser. Das Jupyter-Interface sollte laden.")

        print(f"\n3. 📈 Grafana Dashboard (Monitoring):
  - Web-Adresse: http://{server_ip}:3000
  - Standard-Login: admin / admin
  - Funktionsprüfung: Prüfe, ob das NVIDIA DCGM-Exporter Dashboard Daten (GPU-Auslastung/VRAM) anzeigt.")

        print(f"\n4. 🔍 Prometheus (Metriken-Datenbank):
  - Web-Adresse: http://{server_ip}:9090
  - Funktionsprüfung: Gehe auf 'Status' -> 'Targets' und prüfe, ob 'dcgm-exporter' als 'UP' (grün) markiert ist.")
        print("=" * 65)
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Fehler beim Starten der Container:\n{e.stderr}")

start_stack_and_show_guide()